<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/VerySimpleInputPertubation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [119]:
!pip install transformer_lens

In [120]:
import torch

In [121]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [122]:
### Model

from transformer_lens import HookedTransformer, HookedTransformerConfig

E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    d_mlp=1024,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    normalization_type="LN",
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [123]:
import numpy as np
import pandas as pd
import torch
import ast
from torch.utils.data import Dataset, DataLoader


In [147]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

# Add mappings for special tokens
id_to_entity[210] = ','
id_to_entity[211] = '?'

In [148]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [149]:
### Some very basic checks: What happens if we pertube the input sequence

In [150]:
example, label = test_dataset[0]

In [151]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [152]:
with torch.no_grad():
  logits = model(example)

In [153]:
pred = logits[0, -1, :].argmax().item()
print(f"Pred: {pred} ({id_to_entity.get(pred, f'Unknown Token {pred}')})")

Pred: 105 (Manila)


In [154]:
all_logits = logits[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 105 (Manila) → Logit: 11.5813, Probability: 63.14%
Token 106 (Shanghai) → Logit: 10.4718, Probability: 20.82%
Token 155 (Philadelphia) → Logit: 9.3648, Probability: 6.88%
Token 169 (Cape Town) → Logit: 8.9561, Probability: 4.57%
Token 118 (Istanbul) → Logit: 8.9376, Probability: 4.49%
Token 111 (Sao Paulo) → Logit: 3.7446, Probability: 0.02%
Token 154 (Miami) → Logit: 2.7084, Probability: 0.01%
Token 151 (Houston) → Logit: 2.2000, Probability: 0.01%
Token 129 (Bangkok) → Logit: 1.9961, Probability: 0.00%
Token 113 (Karachi) → Logit: 1.9321, Probability: 0.00%


In [155]:
### Changing the 105 token

In [156]:
example_2 = example.clone()

# Define perturbation parameters
token_index_to_change = 6
new_token_value = 123

original_token_name = id_to_entity.get(example_2[token_index_to_change].item(), f"Unknown Token {example_2[token_index_to_change]}")
new_token_name = id_to_entity.get(new_token_value, f"Unknown Token {new_token_value}")
example_2[token_index_to_change] = new_token_value

print(f"Example with Token {token_index_to_change} perturbation ({original_token_name} to {new_token_name}):")
for token_id in example_2:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example with Token 6 perturbation (Manila to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 123 (Lahore)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [157]:
with torch.no_grad():
  logits_2 = model(example_2)
pred_2 = logits_2[0, -1, :].argmax().item()
print(f"Pred: {pred_2} ({id_to_entity.get(pred_2, f'Unknown Token {pred_2}')})")

Pred: 123 (Lahore)


In [158]:
all_logits = logits_2[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 123 (Lahore) → Logit: 13.5487, Probability: 92.09%
Token 155 (Philadelphia) → Logit: 10.3826, Probability: 3.88%
Token 106 (Shanghai) → Logit: 9.8046, Probability: 2.18%
Token 118 (Istanbul) → Logit: 9.3684, Probability: 1.41%
Token 169 (Cape Town) → Logit: 8.1298, Probability: 0.41%
Token 154 (Miami) → Logit: 4.0813, Probability: 0.01%
Token 111 (Sao Paulo) → Logit: 3.2971, Probability: 0.00%
Token 198 (Incheon) → Logit: 3.0494, Probability: 0.00%
Token 137 (Berlin) → Logit: 2.1183, Probability: 0.00%
Token 136 (Wuhan) → Logit: 2.0805, Probability: 0.00%


In [159]:
### Changing another random token
example_3 = example.clone()

# Define perturbation parameters
token_index_to_change = 10
new_token_value = 123

original_token_name = id_to_entity.get(example_3[token_index_to_change].item(), f"Unknown Token {example_3[token_index_to_change]}")
new_token_name = id_to_entity.get(new_token_value, f"Unknown Token {new_token_value}")
example_3[token_index_to_change] = new_token_value


print(f"Example with Token {token_index_to_change} perturbation ({original_token_name} to {new_token_name}):")
for token_id in example_3:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_3 = model(example_3)
pred_3 = logits_3[0, -1, :].argmax().item()
print(f"Pred: {pred_3} ({id_to_entity.get(pred_3, f'Unknown Token {pred_3}')})")

Example with Token 10 perturbation (Philadelphia to Lahore):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 123 (Lahore)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 105 (Manila)


In [160]:
all_logits = logits_3[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 105 (Manila) → Logit: 12.0045, Probability: 74.91%
Token 118 (Istanbul) → Logit: 10.3926, Probability: 14.95%
Token 106 (Shanghai) → Logit: 9.1468, Probability: 4.30%
Token 169 (Cape Town) → Logit: 8.9909, Probability: 3.68%
Token 123 (Lahore) → Logit: 8.3934, Probability: 2.02%
Token 111 (Sao Paulo) → Logit: 5.0869, Probability: 0.07%
Token 152 (Dallas) → Logit: 2.5295, Probability: 0.01%
Token 154 (Miami) → Logit: 2.4733, Probability: 0.01%
Token 198 (Incheon) → Logit: 2.1818, Probability: 0.00%
Token 168 (Jinan) → Logit: 1.9081, Probability: 0.00%


In [161]:
### What happens if we change the entity or attribute - breaking the match

In [162]:
example_4 = example.clone()

# Define perturbation parameters
token_index_to_change = 4
new_token_value = 48 # was 43 before

original_token_name = id_to_entity.get(example_4[token_index_to_change].item(), f"Unknown Token {example_4[token_index_to_change]}")
new_token_name = id_to_entity.get(new_token_value, f"Unknown Token {new_token_value}")
example_4[token_index_to_change] = new_token_value

print(f"Example with Token {token_index_to_change} perturbation ({original_token_name} to {new_token_name}):")
for token_id in example_4:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_4 = model(example_4)
pred_4 = logits_4[0, -1, :].argmax().item()
print(f"Pred: {pred_4} ({id_to_entity.get(pred_4, f'Unknown Token {pred_4}')})")

Example with Token 4 perturbation (Keith to Cynthia):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 48 (Cynthia)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 106 (Shanghai)


In [163]:
all_logits = logits_4[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 106 (Shanghai) → Logit: 12.5538, Probability: 53.97%
Token 105 (Manila) → Logit: 12.2720, Probability: 40.71%
Token 169 (Cape Town) → Logit: 10.1365, Probability: 4.81%
Token 155 (Philadelphia) → Logit: 7.6193, Probability: 0.39%
Token 118 (Istanbul) → Logit: 6.2121, Probability: 0.10%
Token 151 (Houston) → Logit: 2.3497, Probability: 0.00%
Token 100 (Tokyo) → Logit: 2.1882, Probability: 0.00%
Token 111 (Sao Paulo) → Logit: 2.1623, Probability: 0.00%
Token 198 (Incheon) → Logit: 2.0831, Probability: 0.00%
Token 154 (Miami) → Logit: 1.8944, Probability: 0.00%


In [164]:
# 48 (Cynthia) is used in another fact - and it seems that it now predicts both of these attributes - but 106 (Shanghai) with a larger likelihood
# even though the type relation is wrong -> maybe it has a bias towards examples used later in the sequence?

# Lets try editing again - with an entity thats not in the sequence

In [165]:
example_5 = example.clone()

# Define perturbation parameters
token_index_to_change = 4
new_token_value = 5 # was 43 before

original_token_name = id_to_entity.get(example_5[token_index_to_change].item(), f"Unknown Token {example_5[token_index_to_change]}")
new_token_name = id_to_entity.get(new_token_value, f"Unknown Token {new_token_value}")
example_5[token_index_to_change] = new_token_value


print(f"Example with Token {token_index_to_change} perturbation ({original_token_name} to {new_token_name}):")
for token_id in example_5:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_5 = model(example_5)
pred_5 = logits_5[0, -1, :].argmax().item()
print(f"Pred: {pred_5} ({id_to_entity.get(pred_5, f'Unknown Token {pred_5}')})")

Example with Token 4 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)
Pred: 106 (Shanghai)


In [166]:
all_logits = logits_5[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 106 (Shanghai) → Logit: 12.4918, Probability: 54.01%
Token 105 (Manila) → Logit: 12.2200, Probability: 41.16%
Token 169 (Cape Town) → Logit: 9.9334, Probability: 4.18%
Token 155 (Philadelphia) → Logit: 7.7642, Probability: 0.48%
Token 118 (Istanbul) → Logit: 6.5362, Probability: 0.14%
Token 151 (Houston) → Logit: 2.3992, Probability: 0.00%
Token 111 (Sao Paulo) → Logit: 2.3263, Probability: 0.00%
Token 100 (Tokyo) → Logit: 2.1390, Probability: 0.00%
Token 198 (Incheon) → Logit: 2.0659, Probability: 0.00%
Token 154 (Miami) → Logit: 2.0567, Probability: 0.00%


In [167]:
## almost the same???  -> discuss
# Now if we change the question entity as well, it should change again

In [168]:
example_6 = example.clone()

# Define perturbation parameters for the first change
token_index_1 = 4
new_token_value_1 = 5 # was 43 (Keith) before

# Define perturbation parameters for the second change
token_index_2 = -2 # -2 refers to the second to last token
new_token_value_2 = 5 # was 43 (Keith) before

original_token_name_1 = id_to_entity.get(example_6[token_index_1].item(), f"Unknown Token {example_6[token_index_1]}")
new_token_name_1 = id_to_entity.get(new_token_value_1, f"Unknown Token {new_token_value_1}")

original_token_name_2 = id_to_entity.get(example_6[token_index_2].item(), f"Unknown Token {example_6[token_index_2]}")
new_token_name_2 = id_to_entity.get(new_token_value_2, f"Unknown Token {new_token_value_2}")


example_6[token_index_1] = new_token_value_1
example_6[token_index_2] = new_token_value_2


print(f"Example with Token {token_index_1} perturbation ({original_token_name_1} to {new_token_name_1}) and Token {token_index_2} perturbation ({original_token_name_2} to {new_token_name_2}):")
for token_id in example_6:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

with torch.no_grad():
  logits_6 = model(example_6)
pred_6 = logits_6[0, -1, :].argmax().item()
print(f"Pred: {pred_6} ({id_to_entity.get(pred_6, f'Unknown Token {pred_6}')})")

Example with Token 4 perturbation (Keith to Erica) and Token -2 perturbation (Keith to Erica):
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 5 (Erica)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 5 (Erica)
Token 211 (?)
Pred: 105 (Manila)


In [169]:
all_logits = logits_6[0, -1, :]
all_probs = torch.softmax(all_logits, dim=0)

# Get the top 10 logit values and their indices
top_logits, top_indices = torch.topk(all_logits, 10)

print("Top 10 predictions, logits, and probabilities (softmax over all logits):")
# Iterate through the top indices to get corresponding probabilities
for i in range(len(top_indices)):
    idx = top_indices[i]
    logit = top_logits[i]
    prob = all_probs[idx] # Get probability from the full softmax tensor
    entity_name = id_to_entity.get(idx.item(), f"Unknown Token {idx.item()}")
    print(f"Token {idx.item()} ({entity_name}) → Logit: {logit.item():.4f}, Probability: {prob.item():.2%}")

Top 10 predictions, logits, and probabilities (softmax over all logits):
Token 105 (Manila) → Logit: 12.0737, Probability: 76.24%
Token 118 (Istanbul) → Logit: 10.0716, Probability: 10.30%
Token 106 (Shanghai) → Logit: 9.6761, Probability: 6.93%
Token 155 (Philadelphia) → Logit: 9.3132, Probability: 4.82%
Token 169 (Cape Town) → Logit: 8.2419, Probability: 1.65%
Token 111 (Sao Paulo) → Logit: 3.4405, Probability: 0.01%
Token 129 (Bangkok) → Logit: 2.5360, Probability: 0.01%
Token 154 (Miami) → Logit: 2.4145, Probability: 0.00%
Token 151 (Houston) → Logit: 2.1031, Probability: 0.00%
Token 198 (Incheon) → Logit: 1.9209, Probability: 0.00%
